# 07 — Global Interpretability

## What This Notebook Covers

This is the final analytical notebook — the synthesis. Notebooks 04–06 each examined one model in isolation:

- **Notebook 04** — Decision Tree: decision rules and Gini-based feature importances
- **Notebook 05** — Naïve Bayes: per-class Gaussian means and standard deviations
- **Notebook 06** — KNN: local LIME explanations for individual patients

Here we apply **two model-agnostic techniques** to all three models simultaneously and ask: *across three fundamentally different algorithms, which features matter consistently, which are model-dependent, and what do the divergences reveal?*

| Technique | What It Measures |
|---|---|
| **Permutation Importance** | How much *model performance* (F1) degrades when a feature's information is destroyed by randomly shuffling its values across test patients |
| **SHAP** | How much each feature shifts each individual *predicted probability* away from the average prediction, aggregated across all test patients |

The two techniques are complementary. Permutation importance measures impact on *performance*. SHAP measures impact on *predictions*. A feature can rank differently on each — for instance, a feature that is strongly correlated with another may score low on permutation importance (shuffling it doesn't hurt the model if its twin carries the same signal) but still appear meaningfully in SHAP values.

The final section combines both signals into a **consolidated cross-model feature ranking** — our best evidence for which features are robustly important across all three algorithms and both measurement techniques.

> For plain-English descriptions of each clinical feature and its normal range, refer to the **Feature Glossary in notebook 04**.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_data, split_data
from src.models import train_decision_tree, train_knn, train_naive_bayes
from src.interpret_global import permutation_importances, compute_shap_values, plot_shap_summary
from IPython.display import Image

In [ ]:
X, y = load_data('../data/heart_failure_clinical_records_dataset.csv')
X_train, X_test, y_train, y_test = split_data(X, y)

# Same label mapping as notebooks 04-06 — no full glossary repeated here
FEATURE_LABELS = {
    'age':                      'Age (years)',
    'anaemia':                  'Anaemia',
    'creatinine_phosphokinase': 'CPK Enzyme Level (µg/L)',
    'diabetes':                 'Diabetes',
    'ejection_fraction':        'Ejection Fraction (%)',
    'high_blood_pressure':      'High Blood Pressure',
    'platelets':                'Platelet Count (kiloplatelets/mL)',
    'serum_creatinine':         'Serum Creatinine (mg/dL)',
    'serum_sodium':             'Serum Sodium (mEq/L)',
    'sex':                      'Sex (0=Female, 1=Male)',
    'smoking':                  'Smoking',
}

FEATURE_NAMES  = list(X.columns)                          # raw — required by pipelines and ColumnTransformer
READABLE_NAMES = [FEATURE_LABELS[f] for f in FEATURE_NAMES]

dt  = train_decision_tree(X_train, y_train)
knn = train_knn(X_train, y_train)
nb  = train_naive_bayes(X_train, y_train)

print(f'All three models trained on {len(X_train)} patients.')
print(f'Evaluating on {len(X_test)} held-out test patients.')

---
## 1. Permutation Importance

### How It Works

Permutation importance answers: *"How much does model performance drop if I destroy the information carried by one feature?"* The procedure is applied identically to all three models:

1. Evaluate the model on the test set → record the baseline **weighted F1 score**
2. For each feature in turn:
   - Randomly shuffle that feature's values across all test patients, breaking any statistical relationship between that feature and the outcome
   - Re-evaluate the model on the now-corrupted test set
   - **Importance = baseline F1 − shuffled F1** (how much performance dropped)
3. Repeat the shuffle **30 times** and report the **mean** and **standard deviation** of the F1 drops

**Why weighted F1 and not accuracy?** The dataset has class imbalance (~67% survived, ~33% deceased). A model that predicts 'Survived' for every patient achieves 67% accuracy while being completely useless. Weighted F1 accounts for both precision and recall across both classes and penalises poor performance on the minority class.

### Interpreting the Values

| Score range | Interpretation |
|---|---|
| **Large positive** | Shuffling this feature significantly hurt performance — the model was genuinely relying on it |
| **Near zero** | Shuffling had little effect — the feature was either not used, or its information is redundant with another feature that compensated |
| **Negative** | Shuffling this feature actually *improved* F1 — the model learned a spurious correlation on it. More common on small datasets where overfitting can introduce noise |

**Error bars (±std):** Reflect how stable the importance estimate is across 30 random shuffles. When two features have overlapping error bars, their relative ranking is uncertain — be cautious about over-interpreting small differences between features with similar scores.

In [ ]:
perm_dt  = permutation_importances(dt,  X_test, y_test, FEATURE_NAMES)
perm_knn = permutation_importances(knn, X_test, y_test, FEATURE_NAMES)
perm_nb  = permutation_importances(nb,  X_test, y_test, FEATURE_NAMES)

# Build a side-by-side comparison table
perm_table = pd.DataFrame({'feature': FEATURE_NAMES})
perm_table = perm_table.merge(
    perm_dt[['feature', 'importance_mean']].rename(columns={'importance_mean': 'Decision Tree'}), on='feature')
perm_table = perm_table.merge(
    perm_knn[['feature', 'importance_mean']].rename(columns={'importance_mean': 'KNN (k=7)'}), on='feature')
perm_table = perm_table.merge(
    perm_nb[['feature', 'importance_mean']].rename(columns={'importance_mean': 'Naïve Bayes'}), on='feature')

perm_table = perm_table.set_index('feature').round(4)

# Add average across models and sort — highest average importance first
perm_table['Average F1 Drop'] = perm_table.mean(axis=1).round(4)
perm_table = perm_table.sort_values('Average F1 Drop', ascending=False)

# Replace raw index with readable labels
perm_table.index = [FEATURE_LABELS[f] for f in perm_table.index]
perm_table.index.name = 'Feature'

perm_table

In [ ]:
# Build display copies with readable feature labels for the chart y-axes
def make_readable(df):
    d = df.copy()
    d['feature'] = d['feature'].map(FEATURE_LABELS)
    return d

perm_dt_d  = make_readable(perm_dt)
perm_knn_d = make_readable(perm_knn)
perm_nb_d  = make_readable(perm_nb)

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=False)

for ax, df, title in zip(
    axes,
    [perm_dt_d, perm_knn_d, perm_nb_d],
    ['Decision Tree', 'KNN (k=7)', 'Naïve Bayes']
):
    ax.barh(
        df['feature'][::-1],
        df['importance_mean'][::-1],
        xerr=df['importance_std'][::-1],
        color='steelblue',
        capsize=3
    )
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('F1 drop  (mean ± std over 30 shuffles)')

fig.suptitle('Permutation Feature Importance — All Models', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig('../outputs/permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### Reading the Results

Scan across the three panels and look for two patterns:

**Consistent high-scorers** (long bars in all three panels) — these features carry predictive information that no algorithm can compensate for by using other features. Cross-model agreement here is the strongest signal of genuine predictive relevance.

**Inconsistent scorers** (long bar in one panel, near-zero in another) — these reveal how each model's design interacts with the data structure:

- The **Decision Tree** at `max_depth=3` uses at most 3–4 features for its splits. Every other feature scores near zero on permutation importance — not because those features are irrelevant, but because the tree simply never uses them. Shuffling an unused feature cannot change its output.
- **KNN** incorporates all features into every distance calculation, so more features may show non-trivial importance. However, because all features are weighted equally in the distance metric, individually weak features can dilute the contribution of strong ones.
- **Naïve Bayes** also uses all features on every prediction, but the Gaussian assumption may underestimate the importance of skewed features (like Serum Creatinine) by fitting a poor distributional approximation.

**Negative bars** — if present, indicate features for which the model learned a spurious or misleading pattern. Shuffling randomises the noise, which happens to improve the F1 score. This is a warning sign on small datasets and warrants investigation before any deployment.

---
## 2. SHAP Values

### What SHAP Measures

SHAP (SHapley Additive exPlanations) is rooted in **cooperative game theory**. The core idea: treat each feature as a player contributing to a team outcome (the model's prediction), and use Shapley values — a classical concept from game theory — to fairly distribute the total prediction among all features based on their marginal contributions across all possible feature coalitions.

For each patient and each feature, the SHAP value answers:
> *"How much did this feature shift this patient's predicted probability away from the average predicted probability across all patients?"*

**Convention for all plots below:** SHAP values are computed for **class 1 (Deceased)**.
- A **positive SHAP value** means this feature *increased* the predicted probability of death for this patient relative to the baseline
- A **negative SHAP value** means this feature *decreased* the predicted probability of death (pushed toward survival)

### How to Read the Beeswarm Plot

| Element | Meaning |
|---|---|
| **Each dot** | One test patient |
| **Dot position on X (positive / right)** | This feature increased this patient's predicted probability of death |
| **Dot position on X (negative / left)** | This feature decreased this patient's predicted probability of death (pushed toward survival) |
| **Dot colour: Red** | This patient had a *high* value for this feature |
| **Dot colour: Blue** | This patient had a *low* value for this feature |
| **Vertical ordering** | Features sorted by mean absolute SHAP — the most globally influential feature is at the top |

**Two signature patterns to look for:**

- **Risk factor pattern** (red dots on the right, blue dots on the left): high feature value → predicts more death. Example: Serum Creatinine — the higher the creatinine, the more the model pushes toward the Deceased prediction.
- **Protective factor pattern** (red dots on the left, blue dots on the right): high feature value → predicts more survival. Example: Ejection Fraction — a *higher* ejection fraction is *better* for the heart, so high values push SHAP toward the left (away from Deceased).

**Wide dot spread (large x-range):** The feature has variable, patient-specific impact — it interacts differently across different patient contexts.  
**Tight clustering near zero:** The feature contributes very little to predictions regardless of patient.

### Why Different Explainers for Different Models?

| Model | Explainer | Explanation |
|---|---|---|
| **Decision Tree** | `TreeExplainer` | Computes **exact** Shapley values by traversing the tree structure. Fast and mathematically precise. |
| **KNN, Naïve Bayes** | `KernelExplainer` | Model-agnostic approximation. Masks feature subsets using 50 background training samples and approximates Shapley values via weighted linear regression across ~100 perturbed inputs per test patient. Slower and approximate, but applicable to any model. |

> **Important:** SHAP absolute magnitudes are **not comparable across models** — they depend on each model's predicted probability scale. Rankings within a model are reliable; cross-model magnitude comparisons are not. Section 3 converts values to ranks to make cross-model comparison valid.

In [ ]:
# Decision Tree — TreeExplainer (exact Shapley values)
shap_dt = compute_shap_values(dt, X_train, X_test, model_type='tree')

plot_shap_summary(
    shap_dt, X_test, READABLE_NAMES,
    title='SHAP — Decision Tree  (positive SHAP = pushes toward Deceased)',
    path='../outputs/shap_dt.png'
)
Image('../outputs/shap_dt.png')

#### Decision Tree — What to Look For

Because the tree uses `max_depth=3`, only a small number of features appear in its splits. The SHAP plot reflects this sparsity directly:

- **A small number of features** will show wide, spread-out dot distributions — these are the features the tree actually splits on, and they have large, varying SHAP contributions across patients.
- **Most features will show near-zero SHAP values** for virtually every patient — the tree never split on them, so they contribute nothing to any prediction. Their dot clouds will be compressed against the zero line.
- **Ejection Fraction** should show the protective factor pattern: blue dots (low EF, poor heart function) on the right (pushing toward Deceased), red dots (high EF, healthy heart) on the left (pushing toward Survived).
- **Serum Creatinine** should show the risk factor pattern: red dots (high creatinine, kidney stress) on the right, blue dots (low creatinine) on the left.

This plot is the SHAP-equivalent of the Gini feature importances from notebook 04 — same model, same features, but now showing *direction* and *per-patient variability*, not just a single aggregate score.

In [ ]:
# KNN — KernelExplainer (model-agnostic approximation, 50 background samples)
shap_knn = compute_shap_values(knn, X_train, X_test, model_type='kernel')

plot_shap_summary(
    shap_knn, X_test, READABLE_NAMES,
    title='SHAP — KNN k=7  (positive SHAP = pushes toward Deceased)',
    path='../outputs/shap_knn.png'
)
Image('../outputs/shap_knn.png')

#### KNN — What to Look For

KNN incorporates all features into every distance calculation, so expect **more features to show non-zero SHAP values** compared to the Decision Tree.

- **Wider distribution of importance across features** — features the tree ignored entirely may appear here with meaningful SHAP contributions, because the distance metric gives every feature a voice in every prediction.
- **More noise in the plot** — KernelExplainer is an approximation. Some variability in dot positions is attributable to the approximation method itself, not genuine per-patient signal. Treat individual dot positions as directional, not precise.
- **Cross-model consistency check:** Do the top-ranked features match those from the Decision Tree plot? Agreement on the top 2–3 features is meaningful cross-model signal. Divergence in the mid-rank features is expected and reflects the different ways the two algorithms use features.
- **Feature direction consistency:** The protective/risk factor patterns (direction of red vs. blue dot clusters) should be consistent with the Decision Tree — the biology does not change because the algorithm does. A reversal in direction would be a red flag worth investigating.

In [ ]:
# Naïve Bayes — KernelExplainer (model-agnostic approximation, 50 background samples)
shap_nb = compute_shap_values(nb, X_train, X_test, model_type='kernel')

plot_shap_summary(
    shap_nb, X_test, READABLE_NAMES,
    title='SHAP — Naïve Bayes  (positive SHAP = pushes toward Deceased)',
    path='../outputs/shap_nb.png'
)
Image('../outputs/shap_nb.png')

#### Naïve Bayes — What to Look For

Naïve Bayes processes all features via Gaussian likelihood multiplication, but the Gaussian assumption interacts differently with each feature's real-world distribution.

- **Serum Creatinine** may rank *lower* here than in the Decision Tree or KNN SHAP plots. In reality, creatinine values are right-skewed — a small number of very sick patients have extremely high values. GaussianNB forces a symmetric bell curve on this distribution, which poorly captures the outlier patients who drive much of the mortality signal. The Gaussian approximation is weakest precisely where the signal is strongest.
- **Serum Sodium** may rank *higher* here than in the Decision Tree. Naïve Bayes uses it on every prediction through the Gaussian likelihood, whereas the tree at depth 3 likely never allocated a split to it. The modest but consistent mean difference between classes (seen in notebook 05) accumulates into a non-trivial SHAP contribution across all patients.
- **Binary features** (Anaemia, Diabetes, etc.) are expected to show narrow, low-magnitude SHAP distributions — consistent with the weak class separation observed in notebook 05.

---
## 3. Consolidated Feature Ranking

### Why Rank Instead of Compare Raw Values?

Permutation importance is measured in **F1 drop** (absolute change in F1 score). SHAP values are in **probability units** (shift in predicted class probability). These scales are incompatible — comparing 0.05 permutation importance with 0.12 mean absolute SHAP value is meaningless. Even within the same technique, values differ across models because each model's output probabilities are calibrated differently.

The solution: convert each signal to a **rank within that signal**.
- **Rank 1** = most important feature for this model using this technique
- **Rank 11** = least important

We have **6 signals** in total:
- Permutation importance: Decision Tree, KNN, Naïve Bayes → 3 signals
- SHAP mean absolute value: Decision Tree, KNN, Naïve Bayes → 3 signals

**`avg_rank`** is the arithmetic mean of a feature's rank across all 6 signals. A lower avg_rank means the feature consistently appeared near the top — robustly important regardless of model or measurement technique. A high avg_rank means consistently near the bottom. **High rank variance** (some signals rank it #1, others rank it #10) means the feature is model-dependent — genuinely useful to some algorithms but not others.

In [ ]:
def rank_df(df, score_col, name):
    """Rank features by a score column (rank 1 = highest score)."""
    df = df.copy()
    df[name] = df[score_col].rank(ascending=False).astype(int)
    return df.set_index('feature')[name]

def shap_rank(shap_vals, feature_names, name):
    """Rank features by mean absolute SHAP value (rank 1 = highest)."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    s = pd.Series(mean_abs, index=feature_names)
    return s.rank(ascending=False).astype(int).rename(name)

# Permutation importance ranks — higher F1 drop = rank 1
r_perm_dt  = rank_df(perm_dt,  'importance_mean', 'Perm — DT')
r_perm_knn = rank_df(perm_knn, 'importance_mean', 'Perm — KNN')
r_perm_nb  = rank_df(perm_nb,  'importance_mean', 'Perm — NB')

# SHAP mean absolute value ranks — higher |SHAP| = rank 1
r_shap_dt  = shap_rank(shap_dt,  FEATURE_NAMES, 'SHAP — DT')
r_shap_knn = shap_rank(shap_knn, FEATURE_NAMES, 'SHAP — KNN')
r_shap_nb  = shap_rank(shap_nb,  FEATURE_NAMES, 'SHAP — NB')

ranking = pd.concat([
    r_perm_dt, r_perm_knn, r_perm_nb,
    r_shap_dt, r_shap_knn, r_shap_nb
], axis=1)

ranking['avg_rank'] = ranking.mean(axis=1).round(1)
ranking = ranking.sort_values('avg_rank')

# Replace raw feature names with readable labels
ranking.index = [FEATURE_LABELS[f] for f in ranking.index]
ranking.index.name = 'Feature'

ranking

---
## Key Takeaways

### The Robust Signals — Cross-Model Agreement

**Ejection Fraction (%)** is expected to hold the lowest avg_rank — consistently appearing near the top of every single signal across all three models and both techniques. This is the strongest and most reproducible finding of the entire project:
- The Decision Tree split on it at the root node (highest Gini reduction across the whole dataset)
- Naïve Bayes showed the clearest class mean separation on this feature in notebook 05
- LIME in notebook 06 surfaced it in both high-confidence and borderline cases
- Permutation importance and SHAP here confirm it across all three models

Clinically: an ejection fraction below ~40% (normal: ≥ 55%) is a diagnostic criterion for heart failure with reduced ejection fraction (HFrEF). Three very different ML algorithms, evaluated with two independent measurement techniques, converging on this feature is reassuring evidence that the models learned something real about the underlying physiology — not just a statistical artefact of this particular dataset.

**Serum Creatinine (mg/dL)** is expected to rank near the top for the Decision Tree and KNN, but lower for Naïve Bayes. The reason: creatinine is right-skewed in real clinical data — a small number of very ill patients have extremely elevated values (above 5–9 mg/dL) while most patients are clustered near normal ranges. GaussianNB fits a symmetric bell curve to this distribution, which poorly captures the high-end outliers that carry much of the mortality signal. This is a concrete example of how a model's assumption violations translate into measurable underestimation of feature importance.

### The Model-Dependent Signals

**Serum Sodium (mEq/L)** is likely to show higher importance in KNN and Naïve Bayes than in the Decision Tree. The tree at `max_depth=3` simply runs out of splits — with at most 7 nodes, it can only question the 3–4 most decisive features. KNN and NB, processing all features on every prediction, can extract the modest but consistent signal that sodium carries (lower sodium in deceased patients — hyponatraemia as a complication of heart failure).

**Binary comorbidities** (Anaemia, Diabetes, High Blood Pressure, Sex, Smoking) should cluster at the bottom of the ranking with high avg_ranks across all signals. This is an important clinical finding in its own right: in this dataset and follow-up period, the binary comorbidity indicators add minimal marginal predictive value over and above the continuous physiological measurements. The direct clinical measurements (how well is the heart pumping? how are the kidneys coping?) carry far more prognostic information than the presence or absence of pre-existing conditions.

### When SHAP and Permutation Importance Diverge

For most features in a well-behaved dataset, SHAP and permutation importance should broadly agree — both measure how much a feature matters to the model. When they diverge, the most common culprit is **feature correlation**:

- **High SHAP, low permutation importance:** The feature influences predictions, but when it is shuffled away, the model partially compensates by leaning harder on a correlated feature. The information is not lost — it arrives via a different channel. In this dataset, Ejection Fraction and Serum Creatinine are clinically correlated (cardiorenal syndrome): poor cardiac output reduces kidney perfusion, elevating creatinine. Shuffling one may partially preserve the signal through the other.
- **Low SHAP, high permutation importance:** Rarer — usually indicates a feature that contributes through non-linear interactions that the additive SHAP decomposition distributes across many small per-patient values, each appearing small even though the feature collectively matters.

### Summarising for a Clinical Audience

> *"Across three machine learning algorithms — a Decision Tree, Naïve Bayes, and K-Nearest Neighbours — evaluated with two independent importance measurement techniques (permutation importance and SHAP), two clinical measurements consistently emerged as the strongest predictors of mortality during follow-up: Ejection Fraction and Serum Creatinine. This aligns with the established cardiology literature on heart failure prognosis, where reduced ejection fraction and cardiorenal syndrome are recognised as major determinants of short-term outcomes. A secondary signal from Serum Sodium was detected by KNN and Naïve Bayes. Binary comorbidity indicators added minimal marginal predictive value in this dataset and follow-up window, suggesting that in this cohort, direct physiological measurements outperform comorbidity status as mortality predictors."*

### Limitations of This Analysis

- **Small dataset (~300 patients, ~60 test patients):** Importance estimates from permutation importance and KernelExplainer are noisy at this scale. Features with similar avg_ranks should not be sharply differentiated — error bars overlap substantially.
- **Single train/test split:** All results reflect one fixed 80/20 split (random_state=42). Cross-validated importance estimates would be more statistically robust.
- **KernelExplainer approximation:** KNN and NB SHAP values use 50 background samples and 100 perturbation samples per test instance — conservative settings chosen for speed. Higher sample counts would produce smoother, more stable plots, potentially changing mid-tier rankings.
- **Feature correlation unaddressed:** The consolidated ranking averages over different algorithms' different responses to correlated features, rather than explicitly correcting for correlation structure. A SHAP interaction analysis would be needed to quantify pairwise feature interactions.
- **Gaussian assumption in Naïve Bayes:** As noted throughout, the Gaussian assumption likely underestimates the importance of right-skewed features. This should be taken into account when interpreting Naïve Bayes-specific rankings.